In [1]:
import pandas as pd

# Cargar el dataset indicando que el separador es un punto y coma
df = pd.read_csv('../data/raw/raw-data-kaggle.csv', sep=';')

# Revisar las primeras 5 filas (ahora sí se verá como una tabla real)
df.head()

,athlete,gender,timestamp,distance (m),elapsed time (s),elevation gain (m),average heart rate (bpm)
0,18042525,M,15/12/2019 09:08,2965.8,812,17.4,150.3
1,18042525,M,10/12/2019 19:27,10020.8,3290,52.2,160.8
2,18042525,M,03/12/2019 19:46,12132.2,4027,249.0,148.9
3,18042525,M,26/11/2019 19:46,11631.5,4442,194.0,136.2
4,18042525,M,19/11/2019 19:45,11708.1,4022,250.7,146.0


In [2]:
# 1. Ver cuántas filas y columnas tenemos en total
print("Tamaño del dataset:", df.shape)

# 2. Ver si hay valores nulos o faltantes que debamos limpiar
print("\nValores nulos por columna:")
print(df.isnull().sum())

# 3. Filtrar los datos para nuestro alcance: entrenamientos cortos (ej. menos de 30 minutos / 1800 segundos)
df_cortos = df[df['elapsed time (s)'] <= 1800].copy()
print("\nEntrenamientos cortos (<= 30 min):", len(df_cortos))

Tamaño del dataset: (42116, 7)

Valores nulos por columna:
athlete                         0
gender                        355
timestamp                       0
distance (m)                    0
elapsed time (s)                0
elevation gain (m)              0
average heart rate (bpm)    18384
dtype: int64

Entrenamientos cortos (<= 30 min): 5847


In [3]:
# 1. Calcular el Ritmo (Pace en minutos por kilómetro) para tener nuestra variable de rendimiento
df_cortos['pace_min_km'] = (df_cortos['elapsed time (s)'] / 60) / (df_cortos['distance (m)'] / 1000)

# 2. Decisión de limpieza: eliminar entrenamientos donde falte la frecuencia cardíaca 
# y descartar valores atípicos (por ejemplo, ritmos imposibles menores a 2 min/km o mayores a 10 min/km)
df_clean = df_cortos.dropna(subset=['average heart rate (bpm)']).copy()
df_clean = df_clean[(df_clean['pace_min_km'] >= 2) & (df_clean['pace_min_km'] <= 10)]

print("Tamaño del dataset limpio y listo para analizar:", df_clean.shape)
df_clean.head()

Tamaño del dataset limpio y listo para analizar: (2727, 8)


,athlete,gender,timestamp,distance (m),elapsed time (s),elevation gain (m),average heart rate (bpm),pace_min_km
0,18042525,M,15/12/2019 09:08,2965.8,812,17.4,150.3,4.563131
57,18042525,M,27/02/2019 17:38,3574.7,765,16.6,144.3,3.566733
59,18042525,M,21/02/2019 17:55,4029.4,909,19.0,154.5,3.759865
105,18042525,M,22/03/2018 17:57,4894.7,1301,16.3,143.1,4.429962
114,18042525,M,09/01/2018 19:46,2126.2,726,0.0,123.0,5.690904


In [4]:
import plotly.express as px

# 1. Gráfico de Distribución (Univariado): ¿Cómo se distribuye el ritmo en estos intervalos cortos?
fig1 = px.histogram(df_clean, x='pace_min_km', nbins=40, 
                    title='1. Distribución del Ritmo (Pace) en Entrenamientos Cortos',
                    labels={'pace_min_km': 'Ritmo (min/km)'},
                    color_discrete_sequence=['#1f77b4'])
fig1.show()

# 2. Gráfico de Dispersión (Relación X e Y): ¿Cómo afecta la elevación a las pulsaciones?
fig2 = px.scatter(df_clean, x='elevation gain (m)', y='average heart rate (bpm)', 
                  color='pace_min_km', opacity=0.6,
                  title='2. Impacto de la Elevación (X) en la Frecuencia Cardíaca (Y)',
                  labels={'elevation gain (m)': 'Desnivel Positivo (m)', 
                          'average heart rate (bpm)': 'Frecuencia Cardíaca Promedio (lpm)',
                          'pace_min_km': 'Ritmo (min/km)'})
fig2.show()

# 3. Gráfico de Cajas (Comparación): Diferencias de esfuerzo entre géneros (Contextual)
fig3 = px.box(df_clean, x='gender', y='average heart rate (bpm)', 
              color='gender',
              title='3. Comparación de Frecuencia Cardíaca por Género',
              labels={'gender': 'Género', 'average heart rate (bpm)': 'Pulsaciones (lpm)'})
fig3.show()

In [5]:
# Primero convertimos la columna de texto a formato de fecha real de Python
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'], format='%d/%m/%Y %H:%M')

# Extraemos el mes para agrupar los datos
df_clean['mes'] = df_clean['timestamp'].dt.month

# 4. Gráfico Temporal (Obligatorio por la pauta para la variable T)
# Evolución de la frecuencia cardíaca promedio a lo largo de los meses
df_mensual = df_clean.groupby('mes', as_index=False)['average heart rate (bpm)'].mean()
fig4 = px.line(df_mensual, x='mes', y='average heart rate (bpm)', markers=True,
               title='4. Evolución Temporal: Frecuencia Cardíaca Promedio por Mes',
               labels={'mes': 'Mes del Año (1=Ene, 12=Dic)', 'average heart rate (bpm)': 'Pulsaciones Promedio'})
fig4.show()

# 5. Gráfico de Correlación (Para ver relaciones múltiples rápidamente)
# Cumple con el requisito de "relación entre múltiples variables"
columnas_num = ['distance (m)', 'elevation gain (m)', 'average heart rate (bpm)', 'pace_min_km']
matriz_corr = df_clean[columnas_num].corr()
fig5 = px.imshow(matriz_corr, text_auto=True, aspect="auto",
                 title='5. Matriz de Correlación (Relación entre variables)',
                 color_continuous_scale='Blues')
fig5.show()